# R2 Export - Analysis Data

Exports FantasAI analysis data to Cloudflare R2 for frontend consumption.

## Files Exported

* **breakout_candidates.json** - Weekly breakout candidate predictions
* **sleeper_picks.json** - Undervalued sleeper picks for waiver wire
* **player_news.json** - Recent player news headlines with URLs (top 5 per player)

## Target Paths

* `fantasai/analysis/breakout_candidates.json`
* `fantasai/analysis/sleeper_picks.json`
* `fantasai/analysis/player_news.json`

## Source Tables

* `main.fantasai.export_breakout_candidates`
* `main.fantasai.export_sleeper_picks`
* `main.fantasai.export_player_news`

## Schedule

Runs daily after breakout predictions job completes (approx 08:00 UTC)

## Source Data Queries

This notebook exports data from pre-built export tables. Here are the queries that create those tables:

### 1. Breakout Candidates Query

**Source Table:** `main.fantasai.export_breakout_candidates`

**Current Structure:**
```sql
SELECT 
  player_name,
  position,
  team,
  snap_share_delta,
  opportunity_score,
  avg_snap_share,
  week
FROM main.fantasai.export_breakout_candidates
ORDER BY opportunity_score DESC
```

**Column Definitions:**
* `opportunity_score` - ML-generated breakout probability (higher = more likely to break out)
* `snap_share_delta` - Change in snap share % from previous weeks
* `avg_snap_share` - Average snap share over recent weeks

**Created By:** Breakout Predictions - Weekly Production Run notebook (runs Tuesday 10 AM ET)

---

### 2. Sleeper Picks Query

**Source Table:** `main.fantasai.export_sleeper_picks`

**Current Structure:**
```sql
SELECT 
  player_name,
  position,
  team,
  ownership_pct,      -- ⚠️ ISSUE: Currently 0 for all records
  projected_pts,
  value_score,
  reason
FROM main.fantasai.export_sleeper_picks
ORDER BY value_score DESC
```

**Column Definitions:**
* `ownership_pct` - Sleeper platform ownership percentage (⚠️ needs fix - see below)
* `projected_pts` - Projected fantasy points
* `value_score` - Combined metric of upside/opportunity
* `reason` - Text explanation for the pick

**Known Issue:** All `ownership_pct` values are 0. Table needs rebuild with proper JOIN:

```sql
CREATE OR REPLACE TABLE main.fantasai.export_sleeper_picks AS
SELECT 
    p.player_name,
    p.position,
    p.team,
    COALESCE(o.ownership_pct, 0) as ownership_pct,  -- JOIN to bronze source
    p.projected_pts,
    p.value_score,
    p.reason
FROM main.fantasai.player_projections p
LEFT JOIN main.fantasai.bronze_sleeper_ownership o
    ON p.player_name = o.player_name
WHERE p.value_score > 20  -- Sleeper threshold
ORDER BY value_score DESC
LIMIT 30;
```

---

### Data Quality Checks

**Before exporting, always validate:**
```sql
-- Check for suspicious zero values
SELECT 
  COUNT(*) as total,
  SUM(CASE WHEN ownership_pct = 0 THEN 1 ELSE 0 END) as zero_ownership
FROM main.fantasai.export_sleeper_picks;
```

In [0]:
import json
import boto3
from datetime import datetime
from pyspark.sql import functions as F

# Configuration
CATALOG = "main"
SCHEMA = "fantasai"

# R2 Configuration (using boto3 S3-compatible API)
# Secrets should be stored in Databricks secrets
R2_ACCESS_KEY_ID = dbutils.secrets.get(scope="r2_credentials", key="r2_access_key_id")
R2_SECRET_ACCESS_KEY = dbutils.secrets.get(scope="r2_credentials", key="r2_secret_access_key")
R2_ENDPOINT_URL = dbutils.secrets.get(scope="r2_credentials", key="r2_endpoint_url")
R2_BUCKET_NAME = dbutils.secrets.get(scope="r2_credentials", key="r2_bucket_name")

print(f"✓ Catalog: {CATALOG}")
print(f"✓ Schema: {SCHEMA}")
print(f"✓ R2 Bucket: {R2_BUCKET_NAME}")
print(f"✓ Current time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

In [0]:
# Initialize boto3 S3 client for R2
s3_client = boto3.client(
    's3',
    endpoint_url=R2_ENDPOINT_URL,
    aws_access_key_id=R2_ACCESS_KEY_ID,
    aws_secret_access_key=R2_SECRET_ACCESS_KEY,
    region_name='auto'  # R2 uses 'auto' for region
)

print("✓ R2 client initialized")

In [0]:
print("="*70)
print("EXPORTING BREAKOUT CANDIDATES")
print("="*70)

# Query breakout candidates table
breakout_df = spark.table(f"{CATALOG}.{SCHEMA}.export_breakout_candidates")

record_count = breakout_df.count()
print(f"\nRecords to export: {record_count}")

if record_count > 0:
    # Convert to JSON
    breakout_data = breakout_df.toPandas().to_dict(orient='records')
    
    # Add metadata
    export_payload = {
        "data": breakout_data,
        "metadata": {
            "generated_at": datetime.now().isoformat(),
            "record_count": record_count,
            "source_table": "main.fantasai.export_breakout_candidates"
        }
    }
    
    # Convert to JSON string
    json_string = json.dumps(export_payload, indent=2)
    
    # Upload to R2
    s3_client.put_object(
        Bucket=R2_BUCKET_NAME,
        Key='analysis/breakout_candidates.json',
        Body=json_string.encode('utf-8'),
        ContentType='application/json',
        CacheControl='public, max-age=3600'
    )
    
    print(f"\n✓ Uploaded to R2: fantasai/analysis/breakout_candidates.json")
    print(f"✓ File size: {len(json_string)} bytes")
    print(f"\nSample record:")
    print(json.dumps(breakout_data[0], indent=2))
else:
    print("⚠️ No records to export")

In [0]:
print("\n" + "="*70)
print("EXPORTING SLEEPER PICKS")
print("="*70)

# Query sleeper picks table
sleeper_df = spark.table(f"{CATALOG}.{SCHEMA}.export_sleeper_picks")

record_count = sleeper_df.count()
print(f"\nRecords to export: {record_count}")

if record_count > 0:
    # Convert to JSON
    sleeper_data = sleeper_df.toPandas().to_dict(orient='records')
    
    # Add metadata
    export_payload = {
        "data": sleeper_data,
        "metadata": {
            "generated_at": datetime.now().isoformat(),
            "record_count": record_count,
            "source_table": "main.fantasai.export_sleeper_picks"
        }
    }
    
    # Convert to JSON string
    json_string = json.dumps(export_payload, indent=2)
    
    # Upload to R2
    s3_client.put_object(
        Bucket=R2_BUCKET_NAME,
        Key='analysis/sleeper_picks.json',
        Body=json_string.encode('utf-8'),
        ContentType='application/json',
        CacheControl='public, max-age=3600'
    )
    
    print(f"\n✓ Uploaded to R2: fantasai/analysis/sleeper_picks.json")
    print(f"✓ File size: {len(json_string)} bytes")
    print(f"\nSample record:")
    print(json.dumps(sleeper_data[0], indent=2))
else:
    print("⚠️ No records to export")

In [0]:
print("\n" + "="*70)
print("EXPORTING PLAYER NEWS")
print("="*70)

# Query player news table
news_df = spark.table(f"{CATALOG}.{SCHEMA}.export_player_news")

record_count = news_df.count()
print(f"\nRecords to export: {record_count}")

if record_count > 0:
    # Convert to JSON (convert timestamp to string first)
    news_pandas = news_df.toPandas()
    news_pandas['published_at'] = news_pandas['published_at'].dt.strftime('%Y-%m-%d %H:%M:%S')
    news_data = news_pandas.to_dict(orient='records')
    
    # Add metadata
    export_payload = {
        "data": news_data,
        "metadata": {
            "generated_at": datetime.now().isoformat(),
            "record_count": record_count,
            "total_players": news_df.select("player_id").distinct().count(),
            "max_articles_per_player": 5,
            "data_retention_days": 60,
            "source_table": "main.fantasai.export_player_news"
        }
    }
    
    # Convert to JSON string
    json_string = json.dumps(export_payload, indent=2)
    
    # Upload to R2
    s3_client.put_object(
        Bucket=R2_BUCKET_NAME,
        Key='analysis/player_news.json',
        Body=json_string.encode('utf-8'),
        ContentType='application/json',
        CacheControl='public, max-age=3600'
    )
    
    print(f"\n✓ Uploaded to R2: fantasai/analysis/player_news.json")
    print(f"✓ File size: {len(json_string)} bytes")
    print(f"\nSample record:")
    print(json.dumps(news_data[0], indent=2))
else:
    print("⚠️ No records to export")

In [0]:
print("\n" + "="*70)
print("EXPORT SUMMARY")
print("="*70)

print("\n✅ Export Complete!")
print(f"\nFiles exported to R2 bucket '{R2_BUCKET_NAME}':")
print("  • analysis/breakout_candidates.json")
print("  • analysis/sleeper_picks.json")
print("  • analysis/player_news.json")
print(f"\nGenerated at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S UTC')}")
print("\nPublic URLs:")
print("  • https://api.fantasai.net/api/v1/r2/fantasai/analysis/breakout_candidates.json")
print("  • https://api.fantasai.net/api/v1/r2/fantasai/analysis/sleeper_picks.json")
print("  • https://api.fantasai.net/api/v1/r2/fantasai/analysis/player_news.json")